# 01 — Data Ingestion & Dtype Inference Audit

**Source:** Fudo Metabase export (`SKALAR - Data Revenue_*.csv`)
**Protocol:** *Protocolo de Comunicación de Datos Transaccionales Fudo–Skalar (2026-04-09)*

This notebook:
1. Loads the raw CSV with **zero parsing hints** to capture what pandas infers.
2. Compares inferred dtypes against the canonical schema defined in the protocol.
3. Applies corrective parsing to produce a protocol-aligned DataFrame.
4. Persists the clean output to `data/interim/` for downstream notebooks.

In [1]:
import pandas as pd
import yaml

from fudo.utils.io import RAW_DIR, INTERIM_DIR, CONFIGS_DIR

INTERIM_DIR.mkdir(parents=True, exist_ok=True)

# Resolve the source file (glob for the SKALAR revenue export)
source_files = sorted(RAW_DIR.glob("SKALAR - Data Revenue_*.csv"))
assert source_files, "No SKALAR revenue CSV found in data/raw/"
SOURCE_FILE = source_files[-1]  # latest export
print(f"Source: {SOURCE_FILE.name}")

# Load the protocol schema
with open(CONFIGS_DIR / "dataset_schema.yaml") as f:
    SCHEMA = yaml.safe_load(f)

Source: SKALAR - Data Revenue_2026-04-16T13_53_52.265503937Z.csv


## 1. Naive load — let pandas infer everything

No `dtype`, no `parse_dates`, no `thousands` — raw engine inference.

In [2]:
df_raw = pd.read_csv(SOURCE_FILE, low_memory=False)
print(f"Shape: {df_raw.shape}")
df_raw.head()

Shape: (33959, 7)


,PAIS,a.ACCOUNT_ID,FECHA_PRIMER_PAGO,PRODUCTO,MES,MONEDA,REV_LOCAL
0,Chile,"349,486","April 1, 2026",Saas,"202,604",CLP,"10,400"
1,Colombia,"309,479","September 25, 2025",Saas,"202,601",COP,"131,860"
2,Chile,"349,075","March 30, 2026",Payments,"202,604",CLP,29.41
3,Argentina,"324,599","December 2, 2025",Saas,"202,604",ARS,"62,809.92"
4,Chile,"292,587","July 11, 2025",Saas,"202,601",CLP,"34,200"


### 1.1 Pandas inferred dtypes

In [3]:
inferred = pd.DataFrame({
    "raw_column": df_raw.columns,
    "pandas_dtype": [str(df_raw[c].dtype) for c in df_raw.columns],
    "sample_value": [df_raw[c].iloc[0] for c in df_raw.columns],
    "n_unique": [df_raw[c].nunique() for c in df_raw.columns],
})
inferred

,raw_column,pandas_dtype,sample_value,n_unique
0,PAIS,str,Chile,6
1,a.ACCOUNT_ID,str,"349,486",10764
2,FECHA_PRIMER_PAGO,str,"April 1, 2026",289
3,PRODUCTO,str,Saas,3
4,MES,str,"202,604",4
5,MONEDA,str,CLP,6
6,REV_LOCAL,str,"10,400",9110


### 1.2 Inference vs Protocol — comparison table

The Metabase export introduces thousand-separator commas inside quoted fields.
Pandas sees quoted strings and infers `object` for every column that should be
numeric or date-typed. Below we map each raw column to the protocol expectation
and flag the gap.

In [4]:
# Build the comparison: raw column → protocol canonical name → inferred vs expected dtype
raw_to_canonical = {col["raw_name"]: col for col in SCHEMA["columns"] if col["raw_name"]}
extra_cols = {col["raw_name"]: col for col in SCHEMA.get("extra_columns", [])}

comparison_rows = []
for raw_col in df_raw.columns:
    inferred_dtype = str(df_raw[raw_col].dtype)
    if raw_col in raw_to_canonical:
        spec = raw_to_canonical[raw_col]
        comparison_rows.append({
            "raw_column": raw_col,
            "canonical_name": spec["name"],
            "pandas_inferred": inferred_dtype,
            "protocol_dtype": spec["dtype"],
            "match": inferred_dtype == spec["dtype"],
            "in_protocol": True,
        })
    elif raw_col in extra_cols:
        spec = extra_cols[raw_col]
        comparison_rows.append({
            "raw_column": raw_col,
            "canonical_name": f"({spec['name']})",
            "pandas_inferred": inferred_dtype,
            "protocol_dtype": "N/A — not in protocol",
            "match": None,
            "in_protocol": False,
        })

# Append missing protocol columns (channel, client_segment)
for spec in SCHEMA["columns"]:
    if spec["raw_name"] is None:
        comparison_rows.append({
            "raw_column": "—",
            "canonical_name": spec["name"],
            "pandas_inferred": "MISSING",
            "protocol_dtype": spec["dtype"],
            "match": False,
            "in_protocol": True,
        })

dtype_comparison = pd.DataFrame(comparison_rows)
dtype_comparison.style.map(
    lambda v: "background-color: #f4cccc" if v is False else
              "background-color: #d9ead3" if v is True else "",
    subset=["match"],
)

,raw_column,canonical_name,pandas_inferred,protocol_dtype,match,in_protocol
0,PAIS,(pais),str,N/A — not in protocol,None,False
1,a.ACCOUNT_ID,client_id,str,str,True,True
2,FECHA_PRIMER_PAGO,(fecha_primer_pago),str,N/A — not in protocol,None,False
3,PRODUCTO,product,str,str,True,True
4,MES,transaction_date,str,datetime64[ns],False,True
5,MONEDA,currency,str,str,True,True
6,REV_LOCAL,collections,str,float64,False,True
7,—,channel,MISSING,str,False,True
8,—,client_segment,MISSING,str,False,True


### 1.3 Why pandas infers wrong: the Metabase comma problem

The root cause is **one export artifact**: Metabase applies locale-aware
thousand separators to numeric fields inside quoted CSV values.

| Raw value | Pandas sees | Infers as | Should be (protocol) |
|---|---|---|---|
| `"349,486"` | string `349,486` | `object` | `str` (ID — ok, but commas must be stripped) |
| `"202,601"` | string `202,601` | `object` | `datetime64[ns]` — protocol requires `YYYY-MM-DD` |
| `"10,400"` | string `10,400` | `object` | `float64` = `10400.0` |
| `29.41` | number `29.41` | `float64` | `float64` (correct by accident — no comma needed) |

The `REV_LOCAL` column is particularly deceptive: values **without** thousands
(e.g., `29.41`) parse as float, but values **with** thousands (e.g., `"10,400"`)
parse as string. Because the majority have commas, the whole column falls
back to `object`.

In [5]:
# Demonstrate the mixed-type problem on REV_LOCAL
rev_sample = df_raw["REV_LOCAL"].head(10)
print("Raw values and their Python types after pandas read_csv:\n")
for i, v in enumerate(rev_sample):
    print(f"  row {i}: {v!r:>20s}  →  {type(v).__name__}")

Raw values and their Python types after pandas read_csv:

  row 0:             '10,400'  →  str
  row 1:            '131,860'  →  str
  row 2:              '29.41'  →  str
  row 3:          '62,809.92'  →  str
  row 4:             '34,200'  →  str
  row 5:             '24,450'  →  str
  row 6:          '29,442.15'  →  str
  row 7:             '58,425'  →  str
  row 8:               '86.3'  →  str
  row 9:          '91,583.19'  →  str


## 2. Corrective parsing — align to protocol dtypes

We apply transformations column by column to match the canonical schema.

> **NON-CONFORMANCE: `transaction_date`**
>
> The protocol (§3.1) defines `transaction_date` as a `DATE` in `YYYY-MM-DD`
> format — one row per disbursement event. Fudo delivers `MES`, a monthly
> period (`YYYYMM`), meaning daily transaction dates have been rolled up
> before delivery.
>
> We parse `MES` as `period[M]` because that is the best representation of
> what the source actually contains. However, this **does not conform** to the
> protocol's `datetime64[ns]` (`DATE`) requirement. The protocol dtype is
> `DATE`, not `PERIOD`.
>
> **Implication for Skalar:** Any downstream process that requires a specific
> date (e.g., daily cohort assignment, daily FX conversion, intra-month
> timing analysis) will be forced to **fabricate a date** — either by
> assigning an arbitrary convention (first of month, last of month) or by
> assuming uniform distribution. This is an assumption that Skalar would be
> making on Fudo's data, which **compromises the single-source-of-truth
> principle** established in protocol §4.1: *"Los datos transaccionales se
> originan en Fudo. No existe transferencia de custodia ni ventana donde los
> datos consultados por Skalar divergen de los creados por Fudo."*
>
> By collapsing `transaction_date` to a month, Fudo has transferred the
> burden of date-level interpretation to Skalar — an implicit custody
> transfer that the protocol explicitly forbids.

In [6]:
df = df_raw.copy()

# --- client_id (a.ACCOUNT_ID → str, strip commas) ---
df["client_id"] = df["a.ACCOUNT_ID"].str.replace(",", "", regex=False)

# --- transaction_date (MES → period[M]) ---
# NON-CONFORMANT: protocol requires datetime64[ns] (YYYY-MM-DD).
# Source delivers YYYYMM — we parse as period[M] because that is what
# the data actually represents. This is NOT protocol-compliant.
df["transaction_date"] = (
    df["MES"]
    .str.replace(",", "", regex=False)
    .apply(lambda v: pd.Period(year=int(v[:4]), month=int(v[4:6]), freq="M"))
)

# --- currency (MONEDA → str, already clean) ---
df["currency"] = df["MONEDA"].astype(str)

# --- collections (REV_LOCAL → float64, strip commas) ---
df["collections"] = (
    df["REV_LOCAL"]
    .astype(str)
    .str.replace(",", "", regex=False)
    .astype("float64")
)

# --- product (PRODUCTO → str, already clean) ---
df["product"] = df["PRODUCTO"].astype(str)

# --- Extra columns (not in protocol, kept for reference) ---
df["pais"] = df["PAIS"].astype(str)
df["fecha_primer_pago"] = pd.to_datetime(
    df["FECHA_PRIMER_PAGO"], format="%B %d, %Y"
)

# Drop raw columns, keep canonical + extras
canonical_cols = ["client_id", "transaction_date", "currency", "collections", "product"]
extra_cols_keep = ["pais", "fecha_primer_pago"]
df = df[canonical_cols + extra_cols_keep]

print(f"Shape: {df.shape}")
df.head()

Shape: (33959, 7)


,client_id,transaction_date,currency,collections,product,pais,fecha_primer_pago
0,349486,2026-04,CLP,10400.00,Saas,Chile,2026-04-01
1,309479,2026-01,COP,131860.00,Saas,Colombia,2025-09-25
2,349075,2026-04,CLP,29.41,Payments,Chile,2026-03-30
3,324599,2026-04,ARS,62809.92,Saas,Argentina,2025-12-02
4,292587,2026-01,CLP,34200.00,Saas,Chile,2025-07-11


### 2.1 Post-correction dtype check

In [7]:
post_correction = pd.DataFrame({
    "column": df.columns,
    "dtype_after": [str(df[c].dtype) for c in df.columns],
    "sample": [df[c].iloc[0] for c in df.columns],
    "nulls": [df[c].isnull().sum() for c in df.columns],
})
post_correction

,column,dtype_after,sample,nulls
0,client_id,str,349486,0
1,transaction_date,period[M],2026-04,0
2,currency,str,CLP,0
3,collections,float64,10400.0,0
4,product,str,Saas,0
5,pais,str,Chile,0
6,fecha_primer_pago,datetime64[us],2026-04-01 00:00:00,0


## 3. Persist to interim

Save the cleaned DataFrame for use in notebooks 02 and 03.

Note: `transaction_date` is stored as a string (`"2026-01"`, etc.) in parquet
because `period[M]` is not a native parquet type. Downstream notebooks
reconstruct the Period on load. This column remains **non-conformant** with the
protocol's `DATE (YYYY-MM-DD)` requirement throughout the pipeline.

In [8]:
# Convert period to string for parquet compatibility
df_out = df.copy()
df_out["transaction_date"] = df_out["transaction_date"].astype(str)

output_path = INTERIM_DIR / "fudo_revenue_clean.parquet"
df_out.to_parquet(output_path, index=False)
print(f"Saved → {output_path}  ({output_path.stat().st_size / 1024:.0f} KB)")

Saved → /home/alfonso/projects/fudo/data/interim/fudo_revenue_clean.parquet  (305 KB)
